In [7]:
from pathlib import Path
import polars as pl

#путь к срезу
DATA_PATH = Path(r"C:\Users\cendv\Desktop\nyc_taxi\data\interim\sample_200k.csv")

#загрузка данных
df = pl.read_csv(DATA_PATH)

print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} колонок\n")

#формирование сводной таблицы характеристик
overview_data = []

for col in df.columns:
    dtype = str(df[col].dtype)
    n_unique = df[col].n_unique()
    null_count = df[col].null_count()
    null_pct = round((null_count / df.shape[0]) * 100, 2)

    overview_data.append(
        {
            "Column": col,
            "DataType": dtype,
            "UniqueValues": n_unique,
            "NullCount": null_count,
            "NullPercentage": f"{null_pct}%",
        }
    )

#DataFrame для красивого вывода
overview_df = pl.DataFrame(overview_data)
overview_df


# Поиск нулевых координат
zero_coords = df.filter(
    (pl.col("pickup_longitude") == 0) | 
    (pl.col("pickup_latitude") == 0) | 
    (pl.col("dropoff_longitude") == 0) | 
    (pl.col("dropoff_latitude") == 0)
)
print(f"Количество поездок с нулевыми координатами: {zero_coords.shape[0]}")


# Поиск нулевых координат
zero_coords = df.filter(
    (pl.col("pickup_longitude") == 0) | 
    (pl.col("pickup_latitude") == 0) | 
    (pl.col("dropoff_longitude") == 0) | 
    (pl.col("dropoff_latitude") == 0)
)
print(f"Количество поездок с нулевыми координатами: {zero_coords.shape[0]}")


# Преобразуем строки в datetime и считаем длительность в минутах
df_time = df.with_columns([
    pl.col("tpep_pickup_datetime").str.to_datetime(),
    pl.col("tpep_dropoff_datetime").str.to_datetime()
]).with_columns(
    ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")).dt.total_seconds() / 60).alias("duration_minutes")
)

# Выявим поездки длительностью <= 0 или больше 24 часов (1440 мин)
invalid_duration = df_time.filter(
    (pl.col("duration_minutes") <= 0) | (pl.col("duration_minutes") > 1440)
)
print(f"Поездок с некорректной длительностью: {invalid_duration.shape[0]}")

Размер датасета: 200000 строк, 19 колонок

Количество поездок с нулевыми координатами: 4067
Количество поездок с нулевыми координатами: 4067
Поездок с некорректной длительностью: 233
